# Aion Training v2: Kaggle GPU/TPU Ablation Runner

This notebook runs the 4-way training ablation study on Kaggle's free GPU/TPU accelerator environment:
1. **Condition 1:** SFT Only
2. **Condition 2:** SFT + Verifier Filter
3. **Condition 3:** SFT + GRPO v2
4. **Condition 4:** Full Stack (SFT + Verifier + GRPO v2 + Sandbox)


In [ ]:
# Step 1: Environment Setup & Dependencies
!pip install -q pydantic pyyaml cachetools tree-sitter ruff pytest

In [ ]:
# Step 2: Verify GPU / TPU Device Availability
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Active Accelerator: {device}')
if device == 'cuda':
    print(f'Device Name: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

In [ ]:
# Step 3: Load Data Mixture and Verify Integrity
import json
with open('data/training_v2/manifest.json') as f:
    manifest = json.load(f)
print('Dataset Composition:', manifest['stream_breakdown'])
print('Synthetic Ratio:', manifest['synthetic_ratio'])
assert manifest['synthetic_ratio'] <= 0.50, 'Synthetic cap exceeded!'

In [ ]:
# Step 4: Execute 4-Way Training Ablation Study
from rl.training_v2.ablation_runner import AblationRunner
runner = AblationRunner(val_path='bench_real/splits/val.json')
results = runner.run_all_ablations()
print('Ablation Study Summary:')
for cond, metrics in results['ablations'].items():
    print(f"[{cond}] P={metrics['precision']:.3f} R={metrics['recall']:.3f} F1={metrics['f1']:.3f} FPR={metrics['clean_fpr']:.3f}")